In [1]:
from __future__ import annotations

import gc
import glob
from pathlib import Path
import pickle
import torch

from pykeen.models import RotatE
from pykeen.datasets import EagerDataset
from pykeen.evaluation import RankBasedEvaluator
from pykeen_pipeline import load_config, get_dataset

In [2]:
configs = glob.glob("configs/dataset/disease_protein*.yaml")

for dataset_cfg_path in configs:
    print(f"Processing dataset config: {dataset_cfg_path}")
    # dataset_config = load_config(Path(dataset_cfg_path))

    # data_path = Path("dataset_saves")
    # dataset, dataset_label = get_dataset(data_path, dataset_config)


Processing dataset config: configs/dataset\disease_protein_anatomy.yaml
Processing dataset config: configs/dataset\disease_protein_bioprocess.yaml
Processing dataset config: configs/dataset\disease_protein_cellcomp.yaml
Processing dataset config: configs/dataset\disease_protein_drug.yaml
Processing dataset config: configs/dataset\disease_protein_exposure.yaml
Processing dataset config: configs/dataset\disease_protein_hetero.yaml
Processing dataset config: configs/dataset\disease_protein_homo.yaml
Processing dataset config: configs/dataset\disease_protein_molecular.yaml
Processing dataset config: configs/dataset\disease_protein_pathway.yaml
Processing dataset config: configs/dataset\disease_protein_phenotype.yaml


In [20]:
configs = [
    # 'configs/dataset/disease_protein_anatomy.yaml', 
    # 'configs/dataset/disease_protein_bioprocess.yaml', these are done
    'configs/dataset/disease_protein_cellcomp.yaml', 
    # 'configs/dataset/disease_protein_drug.yaml', 
    # 'configs/dataset/disease_protein_exposure.yaml', 
    # 'configs/dataset/disease_protein_hetero.yaml', 
    # 'configs/dataset/disease_protein_homo.yaml', 
    # 'configs/dataset/disease_protein_molecular.yaml', 
    # 'configs/dataset/disease_protein_pathway.yaml', 
    # 'configs/dataset/disease_protein_phenotype.yaml'
    ]

config = configs[0]
print(f"Processing dataset config: {config}")
dataset_config = load_config(Path(config))

data_path = Path("dataset_saves")
dataset, dataset_label = get_dataset(data_path, dataset_config)

Processing dataset config: configs/dataset/disease_protein_cellcomp.yaml
Loading dataset from dataset_saves\disease_protein_cellcomp.pkl
Train: 503927, Validation: 2000, Test: 16000
Inverse relations - Train: True, Validation: False, Test: False


In [18]:
dataset.testing.relation_labeling

Labeling(label_to_id={'cellcomp_cellcomp': 0, 'cellcomp_protein': 1, 'disease_disease': 2, 'disease_protein': 3, 'protein_protein': 4}, id_to_label={0: 'cellcomp_cellcomp', 1: 'cellcomp_protein', 2: 'disease_disease', 3: 'disease_protein', 4: 'protein_protein'}, _vectorized_mapper=<numpy.vectorize object at 0x0000020F0298DED0>, _vectorized_labeler=<numpy.vectorize object at 0x0000020F0298E050>)

In [2]:
def filter_dataset(
        dataset: EagerDataset, 
        keep_relations: set = {"disease_protein"}, 
        remove_relations: set = None, 
        keep_entities: set = None
    ) -> EagerDataset:

    training_factory = dataset.training

    # Build the relation selection from the original mapping.
    if keep_relations is not None:
        available_relations = set(training_factory.relation_to_id)

        relation_labels = {
            label
            for label in available_relations
            if label in keep_relations
            or (
                label.endswith("_inverse")
                and label.removesuffix("_inverse") in keep_relations
            )
        }
    elif remove_relations:
        relation_labels = {
            label
            for label in training_factory.relation_to_id
            if label not in remove_relations
            and not (
                label.endswith("_inverse")
                and label.removesuffix("_inverse") in remove_relations
            )
        }
    else:
        relation_labels = None

    relation_ids = (
        training_factory.relations_to_ids(relation_labels)
        if relation_labels is not None
        else None
    )

    entity_ids = (
        training_factory.entities_to_ids(keep_entities)
        if keep_entities is not None
        else None
    )

    # Apply exactly the same filtering to all splits.
    return EagerDataset(
        training=dataset.training.new_with_restriction(
            entities=entity_ids,
            relations=relation_ids,
        ),
        validation=dataset.validation.new_with_restriction(
            entities=entity_ids,
            relations=relation_ids,
        ),
        testing=dataset.testing.new_with_restriction(
            entities=entity_ids,
            relations=relation_ids,
        ),
    )


In [5]:
datasets = [
    'disease_protein_anatomy', 
    'disease_protein_bioprocess',
    'disease_protein_cellcomp',
    'disease_protein_drug', 
    'disease_protein_exposure', 
    'disease_protein_hetero', 
    'disease_protein_homo', 
    'disease_protein_molecular', 
    'disease_protein_pathway', 
    'disease_protein_phenotype'
    ]

for dataset_name in datasets:
    dataset_path = f"dataset_saves/{dataset_name}.pkl"

    if Path(dataset_path).exists():
        print(f"Loading dataset from {dataset_path}")
        with Path(dataset_path).open("rb") as f:
            dataset = pickle.load(f)
            dataset = dataset.get_dataset()

    filtered = filter_dataset(dataset)
    print(
        filtered.training.num_triples,
        filtered.validation.num_triples,
        filtered.testing.num_triples,
    )


Loading dataset from dataset_saves/disease_protein_anatomy.pkl
Train: 1967802, Validation: 2000, Test: 16000
Inverse relations - Train: True, Validation: False, Test: False
79695 76 640
Loading dataset from dataset_saves/disease_protein_bioprocess.pkl
Train: 613371, Validation: 2000, Test: 16000
Inverse relations - Train: True, Validation: False, Test: False
77920 269 2222
Loading dataset from dataset_saves/disease_protein_cellcomp.pkl
Train: 503927, Validation: 2000, Test: 16000
Inverse relations - Train: True, Validation: False, Test: False
77537 305 2569
Loading dataset from dataset_saves/disease_protein_drug.pkl
Train: 1820093, Validation: 2000, Test: 16000
Inverse relations - Train: True, Validation: False, Test: False
79651 91 669
Loading dataset from dataset_saves/disease_protein_exposure.pkl
Train: 421266, Validation: 2000, Test: 16000
Inverse relations - Train: True, Validation: False, Test: False
76923 376 3112
Loading dataset from dataset_saves/disease_protein_hetero.pkl
Tra



| Dataset | Train előtt | Validation előtt | Test előtt | Train után | Validation után | Test után |
|---|---:|---:|---:|---:|---:|---:|
| disease_protein_anatomy | 1,967,802 | 2,000 | 16,000 | 79,695 | 76 | 640 |
| disease_protein_bioprocess | 613,371 | 2,000 | 16,000 | 77,920 | 269 | 2,222 |
| disease_protein_cellcomp | 503,927 | 2,000 | 16,000 | 77,537 | 305 | 2,569 |
| disease_protein_drug | 1,820,093 | 2,000 | 16,000 | 79,651 | 91 | 669 |
| disease_protein_exposure | 421,266 | 2,000 | 16,000 | 76,923 | 376 | 3,112 |
| disease_protein_hetero | 415,680 | 2,000 | 16,000 | 76,883 | 377 | 3,151 |
| disease_protein_homo | 70,411 | 2,000 | 8,000 | 70,411 | 2,000 | 8,000 |
| disease_protein_molecular | 498,784 | 2,000 | 16,000 | 77,422 | 338 | 2,651 |
| disease_protein_pathway | 460,795 | 2,000 | 16,000 | 77,203 | 355 | 2,853 |
| disease_protein_phenotype | 589,251 | 2,000 | 16,000 | 77,898 | 265 | 2,248 |



In [ ]:
datasets = [
    'disease_protein_anatomy', 
    'disease_protein_bioprocess',
    'disease_protein_cellcomp',
    'disease_protein_drug', 
    'disease_protein_exposure', 
    'disease_protein_hetero', 
    'disease_protein_homo', 
    'disease_protein_molecular', 
    'disease_protein_pathway', 
    'disease_protein_phenotype'
    ]

for dataset_name in datasets:
    dataset_path = f"dataset_saves/{dataset_name}.pkl"
    run_dir = Path(glob.glob(f'dataset_gridsearch_results/{dataset_name}_*_123')[0])
    run_config = load_config(run_dir / "config.yaml")
    model_path = run_dir / "best_model.pth"
    model_path2 = run_dir / "trained_model.pkl"

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f"Run directory: {run_dir}")
    print(f"Device: {device}")

    if Path(dataset_path).exists():
        print(f"Loading dataset from {dataset_path}")
        with Path(dataset_path).open("rb") as f:
            dataset = pickle.load(f)
            dataset = dataset.get_dataset()

    # model = RotatE(
    #     triples_factory=dataset.training,
    #     **run_config.get("model_kwargs", {}),
    # )
    # model.load_state_dict(torch.load(model_path, map_location="cpu"))

    model = torch.load(
        model_path2,
        map_location="cpu",
        weights_only=False,
    )
    model.to(device)
    model.eval()
    print("Model loaded. Starting evaluation...")

    evaluator = RankBasedEvaluator(filtered=True)

    results_raw = evaluator.evaluate(
        model=model,
        mapped_triples=dataset.testing.mapped_triples,
        additional_filter_triples=[
            dataset.training.mapped_triples,
            dataset.validation.mapped_triples,
        ],
        batch_size=128,
        device=device,
        use_tqdm=True,
    )


    filtered_dataset = filter_dataset(
        dataset=dataset,
        keep_relations={"disease_protein"},
        remove_relations=None,
        keep_entities=None,
    )

    del dataset
    gc.collect()
    if device.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.empty_cache()

    print(
        filtered_dataset.training.num_triples,
        filtered_dataset.validation.num_triples,
        filtered_dataset.testing.num_triples,
    )

    results_filtered = evaluator.evaluate(
        model=model,
        mapped_triples=filtered_dataset.testing.mapped_triples,
        additional_filter_triples=[
            filtered_dataset.training.mapped_triples,
            filtered_dataset.validation.mapped_triples,
        ],
        batch_size=32,
        device=device,
        use_tqdm=True,
    )

    print(dataset_name)
    print("Raw results:")
    print(f"Hits@1: {results_raw.get_metric('hits@1'):.4f}")
    print(f"Hits@10: {results_raw.get_metric('hits@10'):.4f}")
    print(f"Mean Reciprocal Rank: {results_raw.get_metric('mean_reciprocal_rank'):.4f}")

    print("Filtered results:")
    print(f"Hits@1: {results_filtered.get_metric('hits@1'):.4f}")
    print(f"Hits@10: {results_filtered.get_metric('hits@10'):.4f}")
    print(f"Mean Reciprocal Rank: {results_filtered.get_metric('mean_reciprocal_rank'):.4f}")

    del model
    del results_raw
    del results_filtered
    del filtered_dataset
    gc.collect()
    if device.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.empty_cache()


Run directory: dataset_gridsearch_results\disease_protein_cellcomp_RotatE_5f487f67_123
Device: cuda
Loading dataset from dataset_saves/disease_protein_cellcomp.pkl
Train: 503927, Validation: 2000, Test: 16000
Inverse relations - Train: True, Validation: False, Test: False
Model loaded. Starting evaluation...


Evaluating on cuda:0:   0%|          | 0.00/16.0k [00:00<?, ?triple/s]

77537 305 2569


Evaluating on cuda:0:   0%|          | 0.00/2.57k [00:00<?, ?triple/s]

disease_protein_cellcomp
Raw results:
Hits@1: 0.1009
Hits@10: 0.2489
Mean Reciprocal Rank: 0.1510
Filtered results:
Hits@1: 0.3155
Hits@10: 0.5309
Mean Reciprocal Rank: 0.3883
Run directory: dataset_gridsearch_results\disease_protein_exposure_RotatE_b4373784_123
Device: cuda
Loading dataset from dataset_saves/disease_protein_exposure.pkl
Train: 421266, Validation: 2000, Test: 16000
Inverse relations - Train: True, Validation: False, Test: False
Model loaded. Starting evaluation...


...

disease_protein_molecular
Raw results:
Hits@1: 0.1025
Hits@10: 0.2398
Mean Reciprocal Rank: 0.1499
Filtered results:
Hits@1: 0.3342
Hits@10: 0.5419
Mean Reciprocal Rank: 0.4069
Run directory: dataset_gridsearch_results\disease_protein_phenotype_RotatE_982394ac_123
Device: cuda
Loading dataset from dataset_saves/disease_protein_phenotype.pkl
Train: 589251, Validation: 2000, Test: 16000
Inverse relations - Train: True, Validation: False, Test: False
Model loaded. Starting evaluation...


Evaluating on cuda:0:   0%|          | 0.00/16.0k [00:00<?, ?triple/s]

77898 265 2248


Evaluating on cuda:0:   0%|          | 0.00/2.25k [00:00<?, ?triple/s]

disease_protein_phenotype
Raw results:
Hits@1: 0.0890
Hits@10: 0.2235
Mean Reciprocal Rank: 0.1349
Filtered results:
Hits@1: 0.3508
Hits@10: 0.5547
Mean Reciprocal Rank: 0.4201


disease_protein_anatomy
Raw results:
Hits@10: 0.5388
Mean Reciprocal Rank: 0.3381
Filtered results:
Hits@10: 0.3281
Mean Reciprocal Rank: 0.2251

disease_protein_drug
Raw results:
Hits@10: 0.7613
Mean Reciprocal Rank: 0.5740
Filtered results:
Hits@10: 0.6129
Mean Reciprocal Rank: 0.4917

disease_protein_homo
Raw results:
Hits@10: 0.6099
Mean Reciprocal Rank: 0.4877
Filtered results:
Hits@10: 0.6099
Mean Reciprocal Rank: 0.4877

disease_protein_pathway
Raw results:
Hits@10: 0.2702
Mean Reciprocal Rank: 0.1718
Filtered results:
Hits@10: 0.5370
Mean Reciprocal Rank: 0.3860

disease_protein_hetero
Raw results:
Hits@10: 0.2366
Mean Reciprocal Rank: 0.1459
Filtered results:
Hits@10: 0.5381
Mean Reciprocal Rank: 0.3930

disease_protein_bioprocess
Raw results:
Hits@10: 0.2190
Mean Reciprocal Rank: 0.1320
Filtered results:
Hits@10: 0.5072
Mean Reciprocal Rank: 0.3641

disease_protein_cellcomp
Raw results:
Hits@10: 0.2489
Mean Reciprocal Rank: 0.1510
Filtered results:
Hits@10: 0.5309
Mean Reciprocal Rank: 0.3883

disease_protein_exposure
Raw results:
Hits@10: 0.2387
Mean Reciprocal Rank: 0.1478
Filtered results:
Hits@10: 0.5416
Mean Reciprocal Rank: 0.3916

disease_protein_molecular
Raw results:
Hits@10: 0.2398
Mean Reciprocal Rank: 0.1499
Filtered results:
Hits@10: 0.5419
Mean Reciprocal Rank: 0.4069

disease_protein_phenotype
Raw results:
Hits@10: 0.2235
Mean Reciprocal Rank: 0.1349
Filtered results:
Hits@10: 0.5547
Mean Reciprocal Rank: 0.4201


| Dataset | Raw Hits@10 | Raw MRR | Filtered Hits@10 | Filtered MRR |
|---|---:|---:|---:|---:|
| disease_protein_anatomy | 0.5388 | 0.3381 | 0.3281 | 0.2251 |
| disease_protein_drug | 0.7613 | 0.5740 | 0.6129 | 0.4917 |
| disease_protein_homo | 0.6099 | 0.4877 | 0.6099 | 0.4877 |
| disease_protein_pathway | 0.2702 | 0.1718 | 0.5370 | 0.3860 |
| disease_protein_hetero | 0.2366 | 0.1459 | 0.5381 | 0.3930 |
| disease_protein_bioprocess | 0.2190 | 0.1320 | 0.5072 | 0.3641 |
| disease_protein_cellcomp | 0.2489 | 0.1510 | 0.5309 | 0.3883 |
| disease_protein_exposure | 0.2387 | 0.1478 | 0.5416 | 0.3916 |
| disease_protein_molecular | 0.2398 | 0.1499 | 0.5419 | 0.4069 |
| disease_protein_phenotype | 0.2235 | 0.1349 | 0.5547 | 0.4201 |

